# DDXPlus Single-Agent Sequential Baseline

This notebook turns DDXPlus cases into a small sequential diagnostic environment. A single agent starts from demographics plus the `INITIAL_EVIDENCE` group, can request up to `MAX_REQUESTS` additional root evidence fields, and must output a final diagnosis plus a ranked differential in a strict JSON schema.

The notebook supports two modes:

- `RUN_LIVE_API = False`: dry-run preview mode for prompt, schema, and environment validation
- `RUN_LIVE_API = True`: real OpenAI-compatible API evaluation with resume-safe artifact writing


In [ ]:
# import os
# from getpass import getpass

# os.environ["LLM_BASE_URL"] = "https://api.openai.com/v1"
# os.environ["LLM_MODEL"] = "gpt-4.1-mini"

# if not os.environ.get("LLM_API_KEY"):
#     os.environ["LLM_API_KEY"] = getpass("Enter LLM_API_KEY: ")


In [ ]:
from __future__ import annotations

import ast
import json
import os
import random
import re
import subprocess
import sys
import time
import zipfile
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import requests
from IPython.display import display
from tqdm.auto import tqdm

pd.set_option("display.max_colwidth", 140)
pd.set_option("display.max_rows", 20)

ROOT = next(
    (
        candidate
        for candidate in [Path.cwd(), Path.cwd().parent]
        if (candidate / "scripts" / "download_ddxplus.py").exists()
    ),
    Path.cwd(),
)
DATASET_DIR = ROOT / ".data" / "ddxplus" / "22687585"
AUTO_DOWNLOAD_IF_MISSING = True

RUN_LIVE_API = True
ALLOW_DRY_RUN_BENCHMARK = False
RESUME_IF_AVAILABLE = True
MAX_REQUESTS = 222
SEQUENTIAL_SAMPLE_PER_CLASS = 1
SEQUENTIAL_MAX_CASES = 10
RANDOM_SEED = 2919
SPLIT_NAME = "test"
TEMPERATURE = 0.2
RUN_VERSION = "unrestricted_v5_gpt41mini_10casepilot"

LLM_BASE_URL = os.environ.get("LLM_BASE_URL", "https://api.openai.com/v1")
LLM_API_KEY = os.environ.get("LLM_API_KEY", "")
LLM_MODEL = os.environ.get("LLM_MODEL", "gpt-4.1-mini")
USE_JSON_MODE = True  # some OpenAI-compatible providers reject this; keep False unless your endpoint supports it

INPUT_COST_PER_1K_TOKENS = None
OUTPUT_COST_PER_1K_TOKENS = None

run_prefix = "live" if RUN_LIVE_API else "dryrun"
case_suffix = f"_{SEQUENTIAL_MAX_CASES}cases" if SEQUENTIAL_MAX_CASES is not None else ""
RUN_NAME = f"single_agent_{run_prefix}_{SPLIT_NAME}_{SEQUENTIAL_SAMPLE_PER_CLASS}perclass_max{MAX_REQUESTS}{case_suffix}_{RUN_VERSION}"
ARTIFACT_DIR = ROOT / "artifacts" / "sequential_single_agent" / RUN_NAME
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root :", ROOT)
print("Dataset dir  :", DATASET_DIR)
print("Run live API :", LLM_API_KEY)
print("Run version  :", RUN_VERSION)
print("Run name     :", RUN_NAME)
print("Artifact dir :", ARTIFACT_DIR)


## Dataset Helpers

The sequential notebook uses the same DDXPlus file validation and split-loading logic as the one-shot notebook so that `case_id`s remain stable across both baselines.


In [ ]:
REQUIRED_FILES = [
    "release_evidences.json",
    "release_conditions.json",
    "release_train_patients.zip",
    "release_validate_patients.zip",
    "release_test_patients.zip",
]
SPLIT_TO_FILENAME = {
    "train": "release_train_patients.zip",
    "validate": "release_validate_patients.zip",
    "test": "release_test_patients.zip",
}


def ensure_dataset_present(dataset_dir: Path, auto_download: bool = True) -> dict[str, Path]:
    dataset_dir = dataset_dir.expanduser().resolve()
    paths = {name: dataset_dir / name for name in REQUIRED_FILES}
    missing = [name for name, path in paths.items() if not path.exists()]
    if missing and auto_download:
        subprocess.run(
            [sys.executable, str(ROOT / "scripts" / "download_ddxplus.py"), "--output-dir", str(dataset_dir)],
            cwd=ROOT,
            check=True,
        )
        paths = {name: dataset_dir / name for name in REQUIRED_FILES}
        missing = [name for name, path in paths.items() if not path.exists()]
    if missing:
        raise FileNotFoundError(f"Missing dataset files in {dataset_dir}: {', '.join(missing)}")
    invalid_zips = [name for name, path in paths.items() if path.suffix == ".zip" and not zipfile.is_zipfile(path)]
    if invalid_zips:
        raise ValueError(
            f"Invalid or incomplete zip files in {dataset_dir}: {', '.join(invalid_zips)}. "
            "Delete the partial archives and rerun scripts/download_ddxplus.py."
        )
    return paths


def load_json(path: Path) -> dict[str, Any]:
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)


def load_patient_split(zip_path: Path) -> pd.DataFrame:
    with zipfile.ZipFile(zip_path, "r") as archive:
        members = [name for name in archive.namelist() if not name.endswith("/")]
        file_name = next((name for name in members if name.endswith(".csv")), None)
        if file_name is None:
            if not members:
                raise ValueError(f"Archive is empty: {zip_path}")
            file_name = members[0]
        with archive.open(file_name) as handle:
            return pd.read_csv(handle)


def attach_split_metadata(frame: pd.DataFrame, split_name: str) -> pd.DataFrame:
    enriched = frame.copy()
    enriched["source_row_index"] = np.arange(len(enriched), dtype=int)
    enriched["split"] = split_name
    enriched["case_id"] = enriched["source_row_index"].map(lambda idx: f"{split_name}:{idx}")
    return enriched


def _allocate_group_counts(counts: pd.Series, sample_size: int) -> pd.Series:
    if sample_size >= int(counts.sum()):
        return counts.copy()
    raw = counts / counts.sum() * sample_size
    allocated = np.floor(raw).astype(int)
    if sample_size >= len(counts):
        allocated[allocated == 0] = 1
    allocated = np.minimum(allocated, counts.to_numpy())
    deficit = sample_size - int(allocated.sum())
    if deficit > 0:
        order = np.argsort(-(raw - allocated))
        for idx in order:
            if allocated[idx] < counts.iloc[idx]:
                allocated[idx] += 1
                deficit -= 1
            if deficit == 0:
                break
    overflow = int(allocated.sum()) - sample_size
    if overflow > 0:
        order = np.argsort(raw - allocated)
        for idx in order:
            if allocated[idx] > 1:
                allocated[idx] -= 1
                overflow -= 1
            if overflow == 0:
                break
    return pd.Series(allocated, index=counts.index, dtype=int)


def stratified_subset(frame: pd.DataFrame, label_col: str, sample_size: int | None, seed: int) -> pd.DataFrame:
    if sample_size is None or sample_size >= len(frame):
        return frame.reset_index(drop=True)
    counts = frame[label_col].value_counts().sort_index()
    allocations = _allocate_group_counts(counts, sample_size)
    rng = np.random.default_rng(seed)
    sampled_indices = []
    for label, count in allocations.items():
        if count <= 0:
            continue
        label_indices = frame.index[frame[label_col] == label].to_numpy()
        sampled_indices.append(rng.choice(label_indices, size=int(count), replace=False))
    merged = np.concatenate(sampled_indices)
    rng.shuffle(merged)
    return frame.loc[merged].reset_index(drop=True)


def sample_fixed_cases_per_pathology(frame: pd.DataFrame, label_col: str, per_class: int, seed: int) -> pd.DataFrame:
    pieces = []
    for offset, (label, group) in enumerate(frame.groupby(label_col, sort=True)):
        take = min(per_class, len(group))
        pieces.append(group.sample(n=take, random_state=seed + offset))
    sampled = pd.concat(pieces, axis=0).sort_values(["PATHOLOGY", "source_row_index"]).reset_index(drop=True)
    return sampled


## Case Episodes, Evidence Ledger, And Environment

The single-agent workup baseline reuses the case-episode abstraction. The environment exposes the visible evidence ledger plus the remaining unrevealed root evidence questions as the action space.


In [ ]:
def parse_literal_list(raw: Any) -> list[Any]:
    if isinstance(raw, list):
        return raw
    if raw is None:
        return []
    if isinstance(raw, float) and np.isnan(raw):
        return []
    return ast.literal_eval(raw)


def parse_evidence_token(token: str) -> tuple[str, str | None]:
    if "_@_" not in token:
        return token, None
    root_id, value = token.split("_@_", 1)
    return root_id, value


def encode_age(age: int) -> int:
    if age < 1:
        return 0
    if age <= 4:
        return 1
    if age <= 14:
        return 2
    if age <= 29:
        return 3
    if age <= 44:
        return 4
    if age <= 59:
        return 5
    if age <= 74:
        return 6
    return 7


def encode_sex(sex: str) -> int:
    if sex == "M":
        return 0
    if sex == "F":
        return 1
    raise ValueError(f"Unexpected sex value: {sex}")


@dataclass
class ObservationSchema:
    root_ids: list[str]
    slot_slices: dict[str, tuple[int, int]]
    data_types: dict[str, str]
    possible_values: dict[str, list[str]]
    default_values: dict[str, str | None]
    categorical_integer_roots: set[str]
    question_text: dict[str, str]
    feature_names: list[str]

    @classmethod
    def from_metadata(cls, evidence_metadata: dict[str, dict[str, Any]]) -> "ObservationSchema":
        root_ids = list(evidence_metadata.keys())
        slot_slices = {}
        data_types = {}
        possible_values = {}
        default_values = {}
        categorical_integer_roots = set()
        question_text = {}
        feature_names = [f"age_bin_{idx}" for idx in range(8)] + ["sex_M", "sex_F"]
        cursor = 10
        for root_id in root_ids:
            meta = evidence_metadata[root_id]
            data_type = meta.get("data_type", "B")
            raw_values = meta.get("possible-values", [])
            values = [str(value) for value in raw_values]
            default_value = meta.get("default_value")
            default_value = None if default_value is None else str(default_value)
            question_text[root_id] = meta.get("question_en", root_id)
            data_types[root_id] = data_type
            possible_values[root_id] = values
            default_values[root_id] = default_value
            if data_type == "B":
                slot_slices[root_id] = (cursor, cursor + 1)
                feature_names.append(root_id)
                cursor += 1
            elif data_type == "C":
                if raw_values and not isinstance(raw_values[0], str):
                    categorical_integer_roots.add(root_id)
                    slot_slices[root_id] = (cursor, cursor + 1)
                    feature_names.append(root_id)
                    cursor += 1
                else:
                    slot_slices[root_id] = (cursor, cursor + len(values))
                    feature_names.extend(f"{root_id}__{value}" for value in values)
                    cursor += len(values)
            elif data_type == "M":
                slot_slices[root_id] = (cursor, cursor + len(values))
                feature_names.extend(f"{root_id}__{value}" for value in values)
                cursor += len(values)
            else:
                raise ValueError(f"Unsupported evidence type {data_type} for {root_id}")
        return cls(
            root_ids=root_ids,
            slot_slices=slot_slices,
            data_types=data_types,
            possible_values=possible_values,
            default_values=default_values,
            categorical_integer_roots=categorical_integer_roots,
            question_text=question_text,
            feature_names=feature_names,
        )

    @property
    def feature_size(self) -> int:
        return len(self.feature_names)

    def initial_state(self, age: int, sex: str) -> np.ndarray:
        state = np.zeros(self.feature_size, dtype=np.float32)
        state[encode_age(int(age))] = 1.0
        state[8 + encode_sex(sex)] = 1.0
        return state

    def apply_root_observation(self, state: np.ndarray, root_id: str, present_values: list[str] | None = None) -> np.ndarray:
        values = [str(value) for value in (present_values or [])]
        data_type = self.data_types[root_id]
        start, end = self.slot_slices[root_id]
        default_value = self.default_values[root_id]
        if data_type == "B":
            state[start] = 1.0 if values else -1.0
            return state
        if root_id in self.categorical_integer_roots:
            # Assumption: the official one-slot categorical path is preserved by storing
            # unknown=0 and observed category as a normalized scalar in (0, 1].
            chosen = values[0] if values else default_value
            if chosen is None:
                state[start] = 0.0
                return state
            idx = self.possible_values[root_id].index(str(chosen))
            state[start] = float((idx + 1) / max(1, len(self.possible_values[root_id])))
            return state
        state[start:end] = -1.0
        selected_values = values if values else ([default_value] if default_value is not None else [])
        for value in selected_values:
            idx = self.possible_values[root_id].index(str(value))
            state[start + idx] = 1.0
        return state


@dataclass
class EvidenceLedgerEntry:
    root_evidence_id: str
    question_en: str
    source: str
    status: str
    values: list[str] = field(default_factory=list)

    def as_dict(self) -> dict[str, Any]:
        return {
            "root_evidence_id": self.root_evidence_id,
            "question_en": self.question_en,
            "source": self.source,
            "status": self.status,
            "values": list(self.values),
        }


@dataclass
class CaseEpisode:
    case_id: str
    split_name: str
    source_row_index: int
    age: int
    sex: str
    pathology: str
    initial_evidence: str
    differential: list[tuple[str, float]]
    evidence_by_root: dict[str, list[str]]
    visible_evidence: list[str]
    hidden_evidence: list[str]
    requestable_fields: list[str]
    evidence_ledger: list[EvidenceLedgerEntry]
    revealed_roots: set[str]


class CaseEpisodeCompiler:
    def __init__(self, evidence_metadata: dict[str, dict[str, Any]]):
        self.evidence_metadata = evidence_metadata
        self.schema = ObservationSchema.from_metadata(evidence_metadata)
        self.current_episode: CaseEpisode | None = None
        self.parent_question = {
            root_id: str(meta.get("code_question", root_id) or root_id)
            for root_id, meta in evidence_metadata.items()
        }

    def tokens_to_values(self, root_id: str, tokens: list[str]) -> list[str]:
        data_type = self.schema.data_types[root_id]
        if data_type == "B":
            return ["present"] if tokens else []
        return [value for _, value in map(parse_evidence_token, tokens) if value is not None]

    def decode_value(self, root_id: str, value: str) -> str:
        if value == "present":
            return "yes"
        value_meaning = self.evidence_metadata[root_id].get("value_meaning", {})
        if isinstance(value_meaning, dict):
            entry = value_meaning.get(str(value))
            if isinstance(entry, dict):
                human = entry.get("en") or entry.get("fr")
                if human:
                    return str(human)
            elif entry:
                return str(entry)
        return str(value)

    def decode_values(self, root_id: str, values: list[str]) -> list[str]:
        return [self.decode_value(root_id, value) for value in values]

    def summarize_observation(self, root_id: str, values: list[str], status: str) -> str:
        question = self.schema.question_text[root_id]
        if status == "absent":
            return f"{question} -> no"
        if self.schema.data_types[root_id] == "B":
            return f"{question} -> yes"
        decoded_values = self.decode_values(root_id, values)
        value_text = ", ".join(decoded_values) if decoded_values else "observed"
        return f"{question} -> {value_text}"

    def render_ledger_entry(self, entry: EvidenceLedgerEntry) -> str:
        return (
            f"- {entry.root_evidence_id}: "
            f"{self.summarize_observation(entry.root_evidence_id, entry.values, entry.status)} "
            f"[source={entry.source}]"
        )

    def render_visible_findings(self, episode: CaseEpisode) -> str:
        return "\n".join(self.render_ledger_entry(entry) for entry in episode.evidence_ledger)

    def root_present_or_implied_present(self, root_id: str, episode: CaseEpisode) -> bool:
        if root_id in episode.revealed_roots and bool(episode.evidence_by_root.get(root_id, [])):
            return True
        for revealed_root in episode.revealed_roots:
            if (
                self.parent_question.get(revealed_root, revealed_root) == root_id
                and bool(episode.evidence_by_root.get(revealed_root, []))
            ):
                return True
        return False

    def parent_is_satisfied(self, root_id: str, episode: CaseEpisode) -> bool:
        parent_root = self.parent_question.get(root_id, root_id)
        if parent_root == root_id:
            return True
        return self.root_present_or_implied_present(parent_root, episode)

    def from_row(self, row: Any, split_name: str | None = None) -> CaseEpisode:
        evidences_list = [str(token) for token in parse_literal_list(row["EVIDENCES"])]
        differential = parse_literal_list(row["DIFFERENTIAL_DIAGNOSIS"])
        initial_evidence = str(row["INITIAL_EVIDENCE"])
        initial_root, _ = parse_evidence_token(initial_evidence)
        evidence_by_root: dict[str, list[str]] = {}
        for token in evidences_list:
            root_id, _ = parse_evidence_token(token)
            evidence_by_root.setdefault(root_id, []).append(token)
        visible_evidence = list(evidence_by_root.get(initial_root, [initial_evidence]))
        hidden_lookup = set(visible_evidence)
        hidden_evidence = [token for token in evidences_list if token not in hidden_lookup]
        differential_pairs = [(str(pathology), float(score)) for pathology, score in differential]
        split_value = str(row.get("split", split_name or "unknown"))
        source_row_index = int(row.get("source_row_index", -1))
        case_id = str(row.get("case_id", f"{split_value}:{source_row_index}"))
        ledger = [
            EvidenceLedgerEntry(
                root_evidence_id=initial_root,
                question_en=self.schema.question_text[initial_root],
                source="initial_evidence",
                status="present",
                values=self.tokens_to_values(initial_root, visible_evidence),
            )
        ]
        episode = CaseEpisode(
            case_id=case_id,
            split_name=split_value,
            source_row_index=source_row_index,
            age=int(row["AGE"]),
            sex=str(row["SEX"]),
            pathology=str(row["PATHOLOGY"]),
            initial_evidence=initial_evidence,
            differential=differential_pairs,
            evidence_by_root=evidence_by_root,
            visible_evidence=visible_evidence,
            hidden_evidence=hidden_evidence,
            requestable_fields=list(self.schema.root_ids),
            evidence_ledger=ledger,
            revealed_roots={initial_root},
        )
        self.current_episode = episode
        return episode

    def build_initial_state(self, episode: CaseEpisode | None = None) -> np.ndarray:
        current = episode or self.current_episode
        if current is None:
            raise ValueError("No episode is loaded.")
        state = self.schema.initial_state(current.age, current.sex)
        for root_id in current.revealed_roots:
            tokens = current.evidence_by_root.get(root_id, [])
            values = self.tokens_to_values(root_id, tokens)
            self.schema.apply_root_observation(state, root_id, values)
        return state

    def request_evidence(self, root_evidence_id: str, episode: CaseEpisode | None = None) -> list[str]:
        current = episode or self.current_episode
        if current is None:
            raise ValueError("No episode is loaded.")
        if root_evidence_id in current.revealed_roots:
            return current.evidence_by_root.get(root_evidence_id, [])
        revealed_tokens = list(current.evidence_by_root.get(root_evidence_id, []))
        current.revealed_roots.add(root_evidence_id)
        if revealed_tokens:
            current.visible_evidence.extend(revealed_tokens)
            hidden_lookup = set(revealed_tokens)
            current.hidden_evidence = [token for token in current.hidden_evidence if token not in hidden_lookup]
            status = "present"
            values = self.tokens_to_values(root_evidence_id, revealed_tokens)
        else:
            status = "absent"
            values = []
        current.evidence_ledger.append(
            EvidenceLedgerEntry(
                root_evidence_id=root_evidence_id,
                question_en=self.schema.question_text[root_evidence_id],
                source="request",
                status=status,
                values=values,
            )
        )
        return revealed_tokens


def render_ledger(episode: CaseEpisode, compiler: CaseEpisodeCompiler) -> str:
    return compiler.render_visible_findings(episode)


class SequentialDiagnosisEnvironment:
    def __init__(self, compiler: CaseEpisodeCompiler):
        self.compiler = compiler
        self.current_episode: CaseEpisode | None = None

    def reset(self, row: dict[str, Any], split_name: str) -> CaseEpisode:
        self.current_episode = self.compiler.from_row(row, split_name=split_name)
        return self.current_episode

    def available_actions(self, episode: CaseEpisode | None = None) -> list[dict[str, str]]:
        current = episode or self.current_episode
        if current is None:
            raise ValueError("No episode is loaded.")
        available = []
        for root_id in self.compiler.schema.root_ids:
            if root_id in current.revealed_roots:
                continue
            if not self.compiler.parent_is_satisfied(root_id, current):
                continue
            parent_root = self.compiler.parent_question.get(root_id, root_id)
            available.append(
                {
                    "root_evidence_id": root_id,
                    "question_en": self.compiler.schema.question_text[root_id],
                    "parent_root_id": parent_root,
                }
            )
        return available

    def reveal(self, root_evidence_id: str, episode: CaseEpisode | None = None) -> dict[str, Any]:
        current = episode or self.current_episode
        if current is None:
            raise ValueError("No episode is loaded.")
        revealed_tokens = self.compiler.request_evidence(root_evidence_id, current)
        raw_values = self.compiler.tokens_to_values(root_evidence_id, revealed_tokens)
        return {
            "root_evidence_id": root_evidence_id,
            "question_en": self.compiler.schema.question_text[root_evidence_id],
            "status": "present" if revealed_tokens else "absent",
            "revealed_tokens": list(revealed_tokens),
            "revealed_values": raw_values,
            "revealed_value_labels": self.compiler.decode_values(root_evidence_id, raw_values),
            "summary": self.compiler.summarize_observation(
                root_evidence_id,
                raw_values,
                "present" if revealed_tokens else "absent",
            ),
        }

    def visible_context(self, episode: CaseEpisode | None = None) -> dict[str, Any]:
        current = episode or self.current_episode
        if current is None:
            raise ValueError("No episode is loaded.")
        initial_root, _ = parse_evidence_token(current.initial_evidence)
        initial_values = self.compiler.tokens_to_values(initial_root, current.evidence_by_root.get(initial_root, []))
        return {
            "case_id": current.case_id,
            "age": current.age,
            "sex": current.sex,
            "initial_evidence": current.initial_evidence,
            "initial_evidence_summary": self.compiler.summarize_observation(initial_root, initial_values, "present"),
            "visible_evidence": list(current.visible_evidence),
            "visible_findings_text": self.compiler.render_visible_findings(current),
            "evidence_ledger": [
                {
                    **entry.as_dict(),
                    "decoded_values": self.compiler.decode_values(entry.root_evidence_id, entry.values),
                    "summary": self.compiler.summarize_observation(entry.root_evidence_id, entry.values, entry.status),
                }
                for entry in current.evidence_ledger
            ],
        }


## Prompt Contract And OpenAI-Compatible API Adapter

The agent must return a JSON object with:

- `decision`: `request` or `stop`
- `requested_evidence_id`: nullable DDXPlus root evidence id
- `predicted_pathology`: one of the 49 pathologies
- `ranked_differential`: up to 5 ordered pathologies
- `confidence`: float in `[0, 1]`
- `brief_reasoning`: short text


In [ ]:
SEQUENTIAL_RESPONSE_SCHEMA = {
    "decision": "request | stop",
    "requested_evidence_id": "nullable string root evidence id",
    "predicted_pathology": "one of the 49 DDXPlus pathologies",
    "ranked_differential": "ordered list of up to 5 pathology names",
    "confidence": "float between 0 and 1",
    "brief_reasoning": "one short sentence",
}


def build_system_prompt(label_names: list[str]) -> str:
    pathology_text = ", ".join(label_names)
    return (
        "You are a single diagnostic workup agent operating in a DDXPlus environment. "
        "You may request one additional evidence field per turn or stop early if you are ready to finalize. "
        "The visible evidence ledger below has already decoded the structured evidence into readable English. "
        "Use the question text and decoded findings for reasoning; use root_evidence_id values only to choose actions. "
        "Do not ask follow-up questions that depend on a finding that is already absent. "
        "Always return exactly one JSON object and no extra text. "
        "Valid pathology labels are: "
        f"{pathology_text}."
    )


def build_user_prompt(
    compiler: CaseEpisodeCompiler,
    episode: CaseEpisode,
    available_actions: list[dict[str, str]],
    turn_index: int,
) -> str:
    initial_root, _ = parse_evidence_token(episode.initial_evidence)
    initial_values = compiler.tokens_to_values(initial_root, episode.evidence_by_root.get(initial_root, []))
    initial_summary = compiler.summarize_observation(initial_root, initial_values, "present")
    action_lines = [
        f"- {item['root_evidence_id']}: {item['question_en']}" for item in available_actions
    ]
    action_text = "\n".join(action_lines) if action_lines else "- no remaining actions"
    return (
        f"Case ID: {episode.case_id}\n"
        f"Turn: {turn_index}\n"
        f"Demographics: age={episode.age}, sex={episode.sex}\n"
        f"Initial visible finding: {initial_summary}\n\n"
        "Visible evidence ledger (decoded):\n"
        f"{render_ledger(episode, compiler)}\n\n"
        "Available evidence requests (request by root_evidence_id only):\n"
        f"{action_text}\n\n"
        "Return JSON with the following keys only:\n"
        f"{json.dumps(SEQUENTIAL_RESPONSE_SCHEMA, indent=2)}\n\n"
        "Rules:\n"
        "- If decision is request, requested_evidence_id must be one of the available root ids.\n"
        "- If decision is stop, requested_evidence_id must be null.\n"
        "- ranked_differential must be ordered from most likely to least likely and contain at most 5 entries.\n"
        "- predicted_pathology must appear in ranked_differential.\n"
        "- Prefer stopping over asking low-value generic questions once the evidence is already informative.\n"
        "- Use the decoded clinical findings in the ledger, not the raw token ids, to reason about the case.\n"
        f"- Maximum total requests allowed in this case: {MAX_REQUESTS}.\n"
    )


def build_messages(
    label_names: list[str],
    compiler: CaseEpisodeCompiler,
    episode: CaseEpisode,
    available_actions: list[dict[str, str]],
    turn_index: int,
) -> list[dict[str, str]]:
    return [
        {"role": "system", "content": build_system_prompt(label_names)},
        {"role": "user", "content": build_user_prompt(compiler, episode, available_actions, turn_index)},
    ]


def chat_completion_url(base_url: str) -> str:
    cleaned = base_url.rstrip("/")
    if cleaned.endswith("/chat/completions"):
        return cleaned
    return f"{cleaned}/chat/completions"


def extract_content_text(response_payload: dict[str, Any]) -> str:
    message = response_payload["choices"][0]["message"]["content"]
    if isinstance(message, str):
        return message
    if isinstance(message, list):
        pieces = []
        for item in message:
            if isinstance(item, dict) and item.get("type") == "text":
                pieces.append(item.get("text", ""))
        return "".join(pieces)
    raise ValueError(f"Unsupported message content payload: {message!r}")


def call_openai_compatible(messages: list[dict[str, str]]) -> tuple[str, dict[str, Any], dict[str, int]]:
    if not LLM_API_KEY:
        raise ValueError("LLM_API_KEY is empty. Fill the notebook placeholder or export it in the environment.")
    body = {
        "model": LLM_MODEL,
        "messages": messages,
        "temperature": TEMPERATURE,
    }
    if USE_JSON_MODE:
        body["response_format"] = {"type": "json_object"}
    response = requests.post(
        chat_completion_url(LLM_BASE_URL),
        headers={
            "Authorization": f"Bearer {LLM_API_KEY}",
            "Content-Type": "application/json",
        },
        json=body,
        timeout=180,
    )
    response.raise_for_status()
    response_payload = response.json()
    usage = response_payload.get("usage", {})
    token_usage = {
        "input_tokens": int(usage.get("prompt_tokens", 0) or 0),
        "output_tokens": int(usage.get("completion_tokens", 0) or 0),
    }
    return extract_content_text(response_payload), response_payload, token_usage


def parse_json_response(raw_text: str) -> dict[str, Any]:
    cleaned = raw_text.strip()
    match = re.search(r"\{.*\}", cleaned, flags=re.S)
    if not match:
        raise ValueError("No JSON object found in the model response.")
    return json.loads(match.group(0))


def _coerce_ranked_list(value: Any) -> list[str]:
    if value is None:
        return []
    if isinstance(value, list):
        return [str(item).strip() for item in value]
    if isinstance(value, str):
        stripped = value.strip()
        if not stripped:
            return []
        try:
            parsed = ast.literal_eval(stripped)
            if isinstance(parsed, list):
                return [str(item).strip() for item in parsed]
        except Exception:
            pass
        return [item.strip() for item in stripped.split("|") if item.strip()]
    return []


def normalize_agent_response(payload: dict[str, Any], label_names: list[str]) -> dict[str, Any]:
    decision = str(payload.get("decision", "stop")).strip().lower()
    if decision not in {"request", "stop"}:
        decision = "stop"
    requested_evidence_id = payload.get("requested_evidence_id")
    if requested_evidence_id in ("", "null", "None"):
        requested_evidence_id = None
    if requested_evidence_id is not None:
        requested_evidence_id = str(requested_evidence_id).strip()
    predicted_pathology = str(payload.get("predicted_pathology", "")).strip()
    ranked_differential = _coerce_ranked_list(payload.get("ranked_differential"))
    valid_ranked = []
    for item in ranked_differential:
        if item in label_names and item not in valid_ranked:
            valid_ranked.append(item)
    if predicted_pathology in label_names and predicted_pathology not in valid_ranked:
        valid_ranked.insert(0, predicted_pathology)
    if not predicted_pathology and valid_ranked:
        predicted_pathology = valid_ranked[0]
    try:
        confidence = float(payload.get("confidence", 0.0) or 0.0)
    except (TypeError, ValueError):
        confidence = 0.0
    confidence = float(np.clip(confidence, 0.0, 1.0))
    brief_reasoning = str(payload.get("brief_reasoning", "")).strip()[:400]
    return {
        "decision": decision,
        "requested_evidence_id": requested_evidence_id if decision == "request" else None,
        "predicted_pathology": predicted_pathology,
        "ranked_differential": valid_ranked[:5],
        "confidence": confidence,
        "brief_reasoning": brief_reasoning,
    }


def validate_agent_response(
    normalized_response: dict[str, Any],
    available_action_ids: set[str],
    label_names: list[str],
) -> str | None:
    if normalized_response["predicted_pathology"] not in label_names:
        return "predicted_pathology is missing or not in the DDXPlus label set."
    if not normalized_response["ranked_differential"]:
        return "ranked_differential is empty."
    if normalized_response["decision"] == "request":
        requested = normalized_response["requested_evidence_id"]
        if requested is None:
            return "decision=request but requested_evidence_id is null."
        if requested not in available_action_ids:
            return "requested_evidence_id is not currently available."
    return None


def build_repair_messages(
    messages: list[dict[str, str]],
    raw_text: str,
    error_message: str,
) -> list[dict[str, str]]:
    return [
        *messages,
        {"role": "assistant", "content": raw_text},
        {
            "role": "user",
            "content": (
                "Your previous response violated the required JSON contract. "
                f"Problem: {error_message}. "
                "Return one corrected JSON object only."
            ),
        },
    ]


def scripted_dry_run_response(
    episode: CaseEpisode,
    available_actions: list[dict[str, str]],
    turn_index: int,
    label_names: list[str],
) -> dict[str, Any]:
    ranked = [pathology for pathology, _ in episode.differential[:5] if pathology in label_names]
    if not ranked:
        ranked = [episode.pathology]
    if turn_index < min(2, MAX_REQUESTS) and available_actions:
        return {
            "decision": "request",
            "requested_evidence_id": available_actions[0]["root_evidence_id"],
            "predicted_pathology": ranked[0],
            "ranked_differential": ranked,
            "confidence": 0.35 + 0.1 * turn_index,
            "brief_reasoning": "Dry-run preview response. Replace with a live API run for real evaluation.",
        }
    return {
        "decision": "stop",
        "requested_evidence_id": None,
        "predicted_pathology": ranked[0],
        "ranked_differential": ranked,
        "confidence": 0.55,
        "brief_reasoning": "Dry-run preview stop decision.",
    }


def estimate_cost(input_tokens: int, output_tokens: int) -> float | None:
    if INPUT_COST_PER_1K_TOKENS is None or OUTPUT_COST_PER_1K_TOKENS is None:
        return None
    return (
        (input_tokens / 1000.0) * float(INPUT_COST_PER_1K_TOKENS)
        + (output_tokens / 1000.0) * float(OUTPUT_COST_PER_1K_TOKENS)
    )


def get_agent_response_with_repair(
    label_names: list[str],
    compiler: CaseEpisodeCompiler,
    episode: CaseEpisode,
    available_actions: list[dict[str, str]],
    turn_index: int,
) -> dict[str, Any]:
    available_action_ids = {item["root_evidence_id"] for item in available_actions}
    base_messages = build_messages(label_names, compiler, episode, available_actions, turn_index)
    messages = list(base_messages)
    raw_attempts = []
    input_tokens = 0
    output_tokens = 0
    api_calls = 0
    last_error = None

    for attempt_index in range(2):
        if RUN_LIVE_API:
            raw_text, raw_payload, usage = call_openai_compatible(messages)
            api_calls += 1
            input_tokens += usage["input_tokens"]
            output_tokens += usage["output_tokens"]
            raw_attempts.append(
                {
                    "attempt_index": attempt_index,
                    "mode": "live_api",
                    "messages": messages,
                    "raw_text": raw_text,
                    "raw_payload": raw_payload,
                    "usage": usage,
                }
            )
        else:
            simulated = scripted_dry_run_response(episode, available_actions, turn_index, label_names)
            raw_text = json.dumps(simulated, ensure_ascii=False)
            raw_attempts.append(
                {
                    "attempt_index": attempt_index,
                    "mode": "dry_run",
                    "messages": messages,
                    "raw_text": raw_text,
                    "raw_payload": {"simulated": True},
                    "usage": {"input_tokens": 0, "output_tokens": 0},
                }
            )

        try:
            parsed = parse_json_response(raw_text)
            normalized = normalize_agent_response(parsed, label_names)
            validation_error = validate_agent_response(normalized, available_action_ids, label_names)
        except Exception as exc:
            normalized = None
            validation_error = f"Malformed JSON: {exc}"

        if validation_error is None and normalized is not None:
            return {
                "normalized_response": normalized,
                "raw_attempts": raw_attempts,
                "api_calls": api_calls,
                "input_tokens": input_tokens,
                "output_tokens": output_tokens,
                "estimated_cost": estimate_cost(input_tokens, output_tokens),
                "error_flags": [],
            }

        last_error = validation_error
        if attempt_index == 0:
            messages = build_repair_messages(base_messages, raw_text, validation_error)

    fallback_prediction = (
        normalized["predicted_pathology"]
        if normalized is not None and normalized.get("predicted_pathology") in label_names
        else label_names[0]
    )
    fallback_ranked = (
        normalized["ranked_differential"]
        if normalized is not None and normalized.get("ranked_differential")
        else [fallback_prediction]
    )
    return {
        "normalized_response": {
            "decision": "stop",
            "requested_evidence_id": None,
            "predicted_pathology": fallback_prediction,
            "ranked_differential": fallback_ranked,
            "confidence": 0.0,
            "brief_reasoning": f"Forced stop after repair failure: {last_error}",
        },
        "raw_attempts": raw_attempts,
        "api_calls": api_calls,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "estimated_cost": estimate_cost(input_tokens, output_tokens),
        "error_flags": [str(last_error)],
    }


def append_jsonl(path: Path, record: dict[str, Any]) -> None:
    with path.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(record, ensure_ascii=False) + "\n")


def upsert_prediction_row(path: Path, row: dict[str, Any]) -> None:
    new_frame = pd.DataFrame([row])
    if path.exists():
        existing = pd.read_csv(path)
        merged = pd.concat([existing, new_frame], ignore_index=True)
        merged = merged.drop_duplicates(subset=["case_id"], keep="last")
    else:
        merged = new_frame
    merged.to_csv(path, index=False)


def parse_ranked_list(value: Any) -> list[str]:
    if isinstance(value, list):
        return [str(item) for item in value]
    if value is None:
        return []
    try:
        parsed = ast.literal_eval(value) if isinstance(value, str) else value
        if isinstance(parsed, list):
            return [str(item) for item in parsed]
    except Exception:
        pass
    return []


def top_k_accuracy_from_lists(true_labels: list[str], ranked_lists: list[list[str]], k: int) -> float:
    hits = []
    for truth, ranked in zip(true_labels, ranked_lists):
        hits.append(int(truth in ranked[:k]))
    return float(np.mean(hits)) if hits else 0.0


def macro_f1_from_names(true_labels: list[str], predicted_labels: list[str], label_names: list[str]) -> float:
    label_to_index = {label: idx for idx, label in enumerate(label_names)}
    gold = np.array([label_to_index[item] for item in true_labels], dtype=np.int64)
    pred = np.array([label_to_index.get(item, -1) for item in predicted_labels], dtype=np.int64)
    scores = []
    for class_idx in range(len(label_names)):
        tp = np.sum((gold == class_idx) & (pred == class_idx))
        fp = np.sum((gold != class_idx) & (pred == class_idx))
        fn = np.sum((gold == class_idx) & (pred != class_idx))
        precision = tp / max(1, tp + fp)
        recall = tp / max(1, tp + fn)
        scores.append(0.0 if precision + recall == 0 else (2 * precision * recall) / (precision + recall))
    return float(np.mean(scores))


def compute_sequential_metrics(prediction_frame: pd.DataFrame, label_names: list[str]) -> dict[str, Any]:
    if prediction_frame.empty:
        return {}
    ranked_lists = [parse_ranked_list(value) for value in prediction_frame["ranked_differential"]]
    true_labels = prediction_frame["true_pathology"].tolist()
    predicted_labels = prediction_frame["predicted_pathology"].tolist()
    metrics = {
        "accuracy": float(np.mean(prediction_frame["correct"])),
        "top3_accuracy": top_k_accuracy_from_lists(true_labels, ranked_lists, 3),
        "top5_accuracy": top_k_accuracy_from_lists(true_labels, ranked_lists, 5),
        "macro_f1": macro_f1_from_names(true_labels, predicted_labels, label_names),
        "mean_requests": float(prediction_frame["num_requests"].mean()),
        "stop_before_cap_rate": float(
            np.mean(
                (prediction_frame["stop_reason"] == "agent_stop")
                & (prediction_frame["num_requests"] < MAX_REQUESTS)
            )
        ),
        "mean_api_calls": float(prediction_frame["api_calls"].mean()),
        "total_input_tokens": int(prediction_frame["input_tokens"].sum()),
        "total_output_tokens": int(prediction_frame["output_tokens"].sum()),
        "total_estimated_cost": (
            None if prediction_frame["estimated_cost"].isna().all()
            else float(prediction_frame["estimated_cost"].fillna(0.0).sum())
        ),
        "turn_distribution": prediction_frame["num_requests"].value_counts().sort_index().to_dict(),
    }
    return metrics


## Benchmark Setup

The sequential benchmark defaults to a fixed **245-case** stratified sample from the test split: `5` cases per pathology. This keeps API-backed evaluation practical while preserving balanced coverage across the 49 diagnoses.


In [ ]:
dataset_paths = ensure_dataset_present(DATASET_DIR, auto_download=AUTO_DOWNLOAD_IF_MISSING)
evidences = load_json(dataset_paths["release_evidences.json"])
conditions = load_json(dataset_paths["release_conditions.json"])

raw_test = attach_split_metadata(load_patient_split(dataset_paths[SPLIT_TO_FILENAME[SPLIT_NAME]]), SPLIT_NAME)
benchmark_df = sample_fixed_cases_per_pathology(
    raw_test,
    label_col="PATHOLOGY",
    per_class=SEQUENTIAL_SAMPLE_PER_CLASS,
    seed=RANDOM_SEED,
)
if SEQUENTIAL_MAX_CASES is not None:
    benchmark_df = (
        benchmark_df.sample(n=min(SEQUENTIAL_MAX_CASES, len(benchmark_df)), random_state=RANDOM_SEED)
        .sort_values(["PATHOLOGY", "source_row_index"])
        .reset_index(drop=True)
    )

compiler = CaseEpisodeCompiler(evidences)
env = SequentialDiagnosisEnvironment(compiler)
label_names = list(conditions.keys())

benchmark_df[["case_id", "PATHOLOGY", "AGE", "SEX", "INITIAL_EVIDENCE"]].to_csv(
    ARTIFACT_DIR / "benchmark_cases.csv",
    index=False,
)

print("Benchmark size :", len(benchmark_df))
print("Case cap       :", SEQUENTIAL_MAX_CASES)
print("Pathologies    :", benchmark_df["PATHOLOGY"].nunique())
display(benchmark_df["PATHOLOGY"].value_counts().sort_index().head(10))
display(benchmark_df[["case_id", "PATHOLOGY", "AGE", "SEX", "INITIAL_EVIDENCE"]].head(3))


## Dry-Run Prompt Preview

You can run this cell without API credentials. It shows the exact prompt structure, the action space format, and a schema-valid response example. That makes it easy to sanity-check the environment before you spend API credits.


In [ ]:
preview_row = benchmark_df.iloc[0].to_dict()
preview_episode = env.reset(preview_row, split_name=SPLIT_NAME)
preview_actions = env.available_actions(preview_episode)
preview_messages = build_messages(label_names, compiler, preview_episode, preview_actions[:25], turn_index=1)
preview_response = scripted_dry_run_response(preview_episode, preview_actions[:25], turn_index=1, label_names=label_names)

print("=== SYSTEM MESSAGE ===")
print(preview_messages[0]["content"][:1500] + "...")
print("\n=== USER MESSAGE (truncated) ===")
print(preview_messages[1]["content"][:3000] + "...")
print("\n=== SCHEMA-VALID PREVIEW RESPONSE ===")
print(json.dumps(preview_response, indent=2))


## Run The Sequential Baseline

Set `RUN_LIVE_API = True` and fill the API placeholders above when you are ready for a real run. The loop is resume-safe: it will skip any `case_id`s already present in `predictions.csv`.


In [ ]:
resolved_run_config = {
    "run_name": RUN_NAME,
    "run_live_api": RUN_LIVE_API,
    "allow_dry_run_benchmark": ALLOW_DRY_RUN_BENCHMARK,
    "resume_if_available": RESUME_IF_AVAILABLE,
    "max_requests": MAX_REQUESTS,
    "sequential_sample_per_class": SEQUENTIAL_SAMPLE_PER_CLASS,
    "random_seed": RANDOM_SEED,
    "sequential_max_cases": SEQUENTIAL_MAX_CASES,
    "split_name": SPLIT_NAME,
    "llm_base_url": LLM_BASE_URL,
    "llm_model": LLM_MODEL,
    "input_cost_per_1k_tokens": INPUT_COST_PER_1K_TOKENS,
    "output_cost_per_1k_tokens": OUTPUT_COST_PER_1K_TOKENS,
}
with (ARTIFACT_DIR / "resolved_run_config.json").open("w", encoding="utf-8") as handle:
    json.dump(resolved_run_config, handle, indent=2)

predictions_path = ARTIFACT_DIR / "predictions.csv"
traces_path = ARTIFACT_DIR / "traces.jsonl"
raw_api_path = ARTIFACT_DIR / "raw_api_responses.jsonl"

if not RUN_LIVE_API and not ALLOW_DRY_RUN_BENCHMARK:
    print("Benchmark execution skipped. Preview mode is enabled. Set RUN_LIVE_API=True to run the real API-backed baseline.")
else:
    processed_case_ids = set()
    if RESUME_IF_AVAILABLE and predictions_path.exists():
        processed_case_ids = set(pd.read_csv(predictions_path)["case_id"].tolist())
        print(f"Resuming run. Existing completed cases: {len(processed_case_ids)}")

    if RUN_LIVE_API and not LLM_API_KEY:
        raise ValueError("RUN_LIVE_API=True but LLM_API_KEY is empty.")

    started_at = time.time()
    for row in tqdm(list(benchmark_df.to_dict(orient="records")), desc="Sequential cases"):
        case_id = row["case_id"]
        if case_id in processed_case_ids:
            continue

        episode = env.reset(row, split_name=SPLIT_NAME)
        case_trace = []
        total_api_calls = 0
        total_input_tokens = 0
        total_output_tokens = 0
        total_estimated_cost = 0.0
        has_cost = False
        error_flags = []
        final_response = None
        stop_reason = "max_requests_reached"

        for turn_index in range(1, MAX_REQUESTS + 1):
            available_actions = env.available_actions(episode)
            turn_bundle = get_agent_response_with_repair(label_names, compiler, episode, available_actions, turn_index)
            normalized = turn_bundle["normalized_response"]
            final_response = normalized
            total_api_calls += turn_bundle["api_calls"]
            total_input_tokens += turn_bundle["input_tokens"]
            total_output_tokens += turn_bundle["output_tokens"]
            if turn_bundle["estimated_cost"] is not None:
                total_estimated_cost += float(turn_bundle["estimated_cost"])
                has_cost = True
            error_flags.extend(turn_bundle["error_flags"])

            for raw_attempt in turn_bundle["raw_attempts"]:
                append_jsonl(
                    raw_api_path,
                    {
                        "case_id": case_id,
                        "turn_index": turn_index,
                        **raw_attempt,
                    },
                )

            trace_step = {
                "turn_index": turn_index,
                "visible_context_before": env.visible_context(episode),
                "agent_response": normalized,
                "error_flags": list(turn_bundle["error_flags"]),
            }

            if normalized["decision"] == "request" and turn_index <= MAX_REQUESTS:
                reveal_payload = env.reveal(normalized["requested_evidence_id"], episode)
                trace_step["reveal_payload"] = reveal_payload
                stop_reason = "max_requests_reached" if turn_index == MAX_REQUESTS else "requested_more_evidence"
                case_trace.append(trace_step)
                if turn_index == MAX_REQUESTS:
                    break
                continue

            stop_reason = "agent_stop"
            case_trace.append(trace_step)
            break

        prediction_row = {
            "case_id": episode.case_id,
            "source_row_index": episode.source_row_index,
            "split": episode.split_name,
            "AGE": episode.age,
            "SEX": episode.sex,
            "true_pathology": episode.pathology,
            "predicted_pathology": final_response["predicted_pathology"],
            "correct": bool(final_response["predicted_pathology"] == episode.pathology),
            "ranked_differential": json.dumps(final_response["ranked_differential"]),
            "num_requests": sum(1 for step in case_trace if step["agent_response"]["decision"] == "request"),
            "stop_reason": stop_reason,
            "api_calls": total_api_calls,
            "input_tokens": total_input_tokens,
            "output_tokens": total_output_tokens,
            "estimated_cost": (total_estimated_cost if has_cost else np.nan),
            "initial_evidence": episode.initial_evidence,
            "error_flags": json.dumps(error_flags),
        }
        upsert_prediction_row(predictions_path, prediction_row)
        append_jsonl(
            traces_path,
            {
                "case_id": episode.case_id,
                "true_pathology": episode.pathology,
                "predicted_pathology": final_response["predicted_pathology"],
                "stop_reason": stop_reason,
                "num_requests": prediction_row["num_requests"],
                "trace": case_trace,
            },
        )

    prediction_frame = pd.read_csv(predictions_path)
    metrics_payload = {
        "run_name": RUN_NAME,
        "num_cases": int(len(prediction_frame)),
        "runtime_seconds": float(time.time() - started_at),
        "metrics": compute_sequential_metrics(prediction_frame, label_names),
        "notes": (
            "Dry-run benchmark results are for schema and environment validation only."
            if not RUN_LIVE_API
            else "Live API-backed sequential baseline."
        ),
    }
    with (ARTIFACT_DIR / "metrics.json").open("w", encoding="utf-8") as handle:
        json.dump(metrics_payload, handle, indent=2)

    display(prediction_frame.head())
    metrics_payload


## Notes

This notebook is intentionally single-agent only. It gives you the fairer pre-agentic comparator for later multi-agent work, but it does not yet implement multi-agent coordination, evidence-gated planning, or a coordinator/safety split. Those belong in the later multi-agent notebooks and reports.
